# FUTURE_DS_02 — Customer Retention & Churn Analysis

## Future Interns — Data Science & Analytics

### Task 2: Customer Retention & Churn Analysis

This notebook analyses customer data for a subscription-based business to identify churn patterns, retention trends, key churn drivers, customer lifetime behaviour, and high-risk customer segments.

## 1. Business Objective

The objective of this analysis is to understand why customers leave the business and identify customer segments that are more likely to churn.

The analysis focuses on:

- Customer churn patterns
- Customer retention
- Contract and payment behaviour
- Internet service behaviour
- Customer tenure
- Potential churn drivers
- High-risk customer segments
- Retention opportunities

## 2. Tools & Technologies

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Jupyter Notebook
- Power BI

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 3. Load Dataset

In [23]:
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 4. Dataset Overview

In [24]:
print('Number of rows:', df.shape[0])
print('Number of columns:', df.shape[1])
    print('\nColumns:')
print(df.columns.tolist())

IndentationError: unexpected indent (2850125328.py, line 3)

In [ ]:
df.info()

## 5. Data Cleaning

`TotalCharges` contains values that need to be converted to numeric format. Invalid values are converted to missing values and removed.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).copy()

df['SeniorCitizenLabel'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df['ChurnFlag'] = df['Churn'].map({'No': 0, 'Yes': 1})

print('Cleaned dataset shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

## 6. Key Performance Indicators

In [ ]:
total_customers = len(df)
churned_customers = df['ChurnFlag'].sum()
retained_customers = total_customers - churned_customers

churn_rate = churned_customers / total_customers * 100
retention_rate = retained_customers / total_customers * 100
average_monthly_charges = df['MonthlyCharges'].mean()
average_total_charges = df['TotalCharges'].mean()
average_tenure = df['tenure'].mean()

kpis = pd.DataFrame({
    'Metric': [
        'Total Customers',
        'Churned Customers',
        'Retained Customers',
        'Churn Rate',
        'Retention Rate',
        'Average Monthly Charges',
        'Average Total Charges',
        'Average Customer Tenure'
    ],
    'Value': [
        total_customers,
        churned_customers,
        retained_customers,
        f'{churn_rate:.2f}%',
        f'{retention_rate:.2f}%',
        f'${average_monthly_charges:,.2f}',
        f'${average_total_charges:,.2f}',
            f'{average_tenure:.2f} months'
    ]
})

kpis

## 7. Churn Rate by Contract Type

Contract type is one of the strongest indicators of customer churn.

In [ ]:
contract_churn = (
    df.groupby('Contract')['ChurnFlag']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

contract_churn

In [ ]:
plt.figure(figsize=(9, 6))
contract_churn.plot(kind='bar')
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** Month-to-month customers have the highest churn rate at approximately 42.71%, while two-year contract customers have the lowest churn rate.

## 8. Churn Rate by Payment Method

In [ ]:
payment_churn = (
    df.groupby('PaymentMethod')['ChurnFlag']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

payment_churn

In [ ]:
plt.figure(figsize=(10, 6))
payment_churn.plot(kind='bar')
plt.title('Churn Rate by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Insight:** Customers using electronic checks have the highest churn rate at approximately 45.29%.

## 9. Churn Rate by Internet Service

In [ ]:
internet_churn = (
    df.groupby('InternetService')['ChurnFlag']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

internet_churn

In [ ]:
plt.figure(figsize=(9, 6))
internet_churn.plot(kind='bar')
plt.title('Churn Rate by Internet Service')
plt.xlabel('Internet Service')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** Fiber optic customers have the highest churn rate among internet service categories at approximately 41.89%.

## 10. Churn Rate by Senior Citizen Status

In [ ]:
senior_churn = (
    df.groupby('SeniorCitizenLabel')['ChurnFlag']
    .mean()
    .mul(100)
)

senior_churn

In [ ]:
plt.figure(figsize=(8, 6))
senior_churn.plot(kind='bar')
plt.title('Churn Rate by Senior Citizen Status')
plt.xlabel('Senior Citizen')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** Senior citizens have a higher churn rate than non-senior customers in this dataset.

## 11. Customer Tenure Analysis

In [ ]:
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months'],
    include_lowest=True
)

tenure_churn = (
    df.groupby('TenureGroup', observed=True)['ChurnFlag']
    .mean()
    .mul(100)
)

tenure_churn

In [ ]:
plt.figure(figsize=(9, 6))
tenure_churn.plot(kind='bar')
plt.title('Churn Rate by Customer Tenure')
plt.xlabel('Tenure Group')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** Customers with 0–12 months of tenure have the highest churn rate at approximately 47.68%. Churn decreases substantially as customer tenure increases.

## 12. Monthly Charges and Churn

In [ ]:
plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges')
plt.title('Monthly Charges by Churn Status')
plt.xlabel('Churn')
plt.ylabel('Monthly Charges')
plt.tight_layout()
plt.show()

The distribution of monthly charges can help identify whether customers who leave the business tend to have different monthly billing levels from retained customers.

## 13. Customer Lifetime Behaviour

In [ ]:
plt.figure(figsize=(9, 6))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True)
plt.title('Customer Tenure Distribution by Churn Status')
plt.xlabel('Tenure (Months)')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

## 14. Potential Churn Drivers

The following variables are analysed to identify groups with higher churn rates.

In [ ]:
driver_columns = [
    'Contract',
    'InternetService',
    'PaymentMethod',
    'TechSupport',
    'OnlineSecurity',
    'PaperlessBilling'
]

for column in driver_columns:
    print('\n' + column)
    result = (
        df.groupby(column)['ChurnFlag']
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )
    display(result.to_frame('Churn Rate (%)'))

## 15. High-Risk Customer Segment

In [ ]:
high_risk = df[
    (df['Contract'] == 'Month-to-month') &
    (df['tenure'] <= 12) &
    (df['MonthlyCharges'] >= df['MonthlyCharges'].median())
].copy()

print('High-risk customers:', len(high_risk))
high_risk.head()

The high-risk segment contains customers who are on month-to-month contracts, have relatively short tenure, and have monthly charges at or above the customer median.

## 16. Key Business Insights

1. The overall churn rate is **26.58%**, meaning more than one-quarter of customers in the cleaned dataset have churned.

2. **Month-to-month contracts** have the highest churn rate at **42.71%**, making contract commitment an important retention factor.

3. Customers using **electronic checks** have the highest churn rate among payment methods at **45.29%**.

4. **Fiber optic customers** have a relatively high churn rate of **41.89%**.

5. Customers with **0–12 months of tenure** have the highest churn rate at **47.68%**.

6. Customers with **TechSupport** and **OnlineSecurity** have substantially lower churn rates than customers without these services.

7. A high-risk segment of **816 customers** was identified using contract type, tenure and monthly charges.

## 17. Retention Recommendations

### 1. Target new customers early
Introduce onboarding campaigns and proactive customer support during the first 12 months, when churn risk is highest.

### 2. Encourage longer contracts
Offer incentives or benefits to month-to-month customers who move to one-year or two-year contracts.

### 3. Investigate electronic-check customers
Encourage customers using electronic checks to consider alternative automatic payment methods where appropriate.

### 4. Improve support and security adoption
Promote TechSupport and OnlineSecurity services because customers using these services show lower churn rates in this dataset.

### 5. Monitor high-risk customers
Create targeted retention campaigns for customers with short tenure, month-to-month contracts and higher monthly charges.

### 6. Investigate fiber-optic churn
Further investigate pricing, service quality, customer support and satisfaction among fiber-optic customers.

## 18. Conclusion

The analysis shows that customer churn is strongly associated with contract type, tenure, payment method and selected service characteristics. The highest-risk customers are generally newer customers with month-to-month contracts and higher monthly charges.

The findings can be used to develop targeted retention strategies, improve onboarding, encourage longer-term contracts and proactively support customers who display high-risk characteristics.

**Project:** FUTURE_DS_02  
**Task:** Customer Retention & Churn Analysis  
**Organisation:** Future Interns — Data Science & Analytics